---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-35: LangChain <b>Text Splitters (Chunking)</b></h1>

# Learning agenda of this notebook

> **The idea of this lecture:** a loader (Lec-34) gives us whole files and whole pages. Before we can embed and search them, we cut them into small pieces called **chunks**. A **text splitter** decides where to cut. Good cuts keep each idea together; bad cuts break sentences apart or mix topics.

1. **Why Split Documents?**
    - Example 1: An embedding model reads only the first 256 tokens
2. **The Two Settings: `chunk_size` and `chunk_overlap`**
3. **Splitting by Length**
    - Example 2: `CharacterTextSplitter`: by paragraphs and by words
    - Example 3: `TokenTextSplitter`: counting tokens instead of characters
4. **Splitting by Text Structure: the Recommended Default**
    - Example 4: `RecursiveCharacterTextSplitter`
    - Example 5: `split_documents()`: chunks that remember their page
5. **Splitting by Document Structure**
    - Example 6: `MarkdownHeaderTextSplitter`: headings become metadata
    - Example 7: A PDF with a complex table: Docling
    - Example 8: Splitting source code
6. **Splitting by Meaning**
    - Example 9: `SemanticChunker`
    - Example 10: Asking an LLM where to cut
7. **Which Splitter Should I Use?**
8. **Summary**

> **Note:** Examples 1 to 9 run locally and need no API key; the first run of Example 7 downloads Docling's models (about 500 MB). Example 10 uses the free `openai/gpt-oss-120b` model on Groq.

# <span style='background :lightgreen' >1. Why Split Documents?</span>

Splitting is the second step of preparing documents for a RAG app, right after loading:

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

Three reasons to split:
1. **Embedding models have a limit.** `all-MiniLM-L6-v2`, the free model we use for embeddings in Lec-36, reads at most **256 tokens**. Anything after that is silently ignored (Example 1).
2. **One vector per idea.** One vector for a whole page mixes many topics, so it matches none of them well. A small chunk about one idea gives a sharp vector that a question can find.
3. **Only the useful part goes into the prompt.** In Lec-38 we paste the best chunks into the prompt. Small chunks mean fewer tokens, lower cost and less noise for the model.

But chunks that are **too small** lose their context: "He holds a Ph.D." means nothing if the chunk does not say who "he" is. Choosing a splitter is choosing a balance.

> **Try it:** ChunkViz (https://chunkviz.up.railway.app/) shows the chunks of any text in colour while you change the settings.

## Setup
We load the biography from Lec-34 and the free embedding model. Both run on your computer.

In [1]:
import numpy as np                                                   # for the cosine similarity
import transformers                                                  # the library under sentence-transformers
from transformers import AutoTokenizer                               # to count tokens the way the model does
from langchain_huggingface import HuggingFaceEmbeddings              # the free local embedding model (Lec-30, Example 14)
from langchain_community.document_loaders import TextLoader          # Lec-34

transformers.logging.set_verbosity_error()                           # hide a warning about long inputs: Example 1 shows it ourselves
transformers.logging.disable_progress_bar()                          # hide the "Loading weights" progress bar

bio = TextLoader("../data/arif_bio.txt", encoding="utf-8").load()[0].page_content   # one Document, we keep its text
print("bio:", len(bio), "characters,", bio.count("\n\n") + 1, "paragraphs")

MODEL = "sentence-transformers/all-MiniLM-L6-v2"                     # small, fast, free: 384 numbers per text
embeddings = HuggingFaceEmbeddings(model_name=MODEL)                 # downloads the model once, then runs locally
tokenizer = AutoTokenizer.from_pretrained(MODEL)                     # the same tokenizer the model uses

def count_tokens(text: str) -> int:
    """Number of tokens the embedding model sees, including its 2 special tokens [CLS] and [SEP]."""
    return len(tokenizer(text)["input_ids"])

def cosine(a, b) -> float:
    """Cosine similarity of two vectors: 1.0 means they point in exactly the same direction."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

bio: 1692 characters, 4 paragraphs


## <span style='color:#7e22ce'>Example 1: An embedding model reads only the first 256 tokens</span>

If the model really read the whole text, any change to the text would change its vector. We add one sentence at the **end** of the biography, and the same sentence at the **start**, and compare the vectors with the original.

In [2]:
print("tokens in the bio       :", count_tokens(bio))                          # more than the model can read
print("the model reads at most :", embeddings._client.max_seq_length, "tokens") # _client is the SentenceTransformer inside

extra = "He also loves playing cricket on weekends."                            # a sentence about something new
at_end = bio + "\n\n" + extra                                                   # the same bio, one sentence longer at the end
at_start = extra + "\n\n" + bio                                                 # the same sentence, but at the start

v_bio, v_end, v_start = embeddings.embed_documents([bio, at_end, at_start])     # three vectors of 384 numbers

print(f"\nsentence added at the END  : similarity {cosine(v_bio, v_end):.4f}")  # 1.0000: the model never saw it
print(f"sentence added at the START: similarity {cosine(v_bio, v_start):.4f}")  # lower: now it is inside the 256 tokens

tokens in the bio       : 315
the model reads at most : 256 tokens

sentence added at the END  : similarity 1.0000
sentence added at the START: similarity 0.9648


**What happened:** the sentence at the end changed nothing: similarity **1.0000**. The model stopped reading after 256 tokens, so the last part of the biography and our new sentence were never embedded. A question about that last part would never find this vector. Splitting the text into chunks that each fit in the limit solves this.

# <span style='background :lightgreen' >2. The Two Settings: <code>chunk_size</code> and <code>chunk_overlap</code></span>

Every splitter has two main settings:
- **`chunk_size`**: the maximum length of a chunk. It is measured by `length_function`: `len` (characters) by default, or a token counter.
- **`chunk_overlap`**: how much of the end of one chunk is repeated at the start of the next one, so that an idea cut at the border is still whole in one of the chunks.

<div style="text-align:center;">
    <img src="../images/langchain-new-22.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**Three ways to call a splitter:**

| Method | Input | Output | Use it for |
|---|---|---|---|
| `split_text(text)` | one `str` | `list[str]` | quick experiments |
| `create_documents([text, ...])` | a list of `str` | `list[Document]` | text you have as strings |
| `split_documents(docs)` | `list[Document]` from a loader | `list[Document]` | **a RAG app**: each chunk keeps the metadata of its Document |

Set `add_start_index=True` and each chunk also records where it starts in the original text.

# <span style='background :lightgreen' >3. Splitting by Length</span>

## <span style='color:#7e22ce'>Example 2: <code>CharacterTextSplitter</code>: by paragraphs and by words</span>

`CharacterTextSplitter` cuts at **one separator** and then joins the pieces back together up to `chunk_size`.
- With `separator="\n\n"` (the default) it cuts at paragraph ends.
- It **never cuts inside a piece**: if one paragraph is longer than `chunk_size`, it becomes a chunk that is too long, with a warning.

In [3]:
from langchain_text_splitters import CharacterTextSplitter    # cuts at one separator only

by_paragraph = CharacterTextSplitter(
    separator="\n\n",       # cut at blank lines: paragraph ends (this is the default)
    chunk_size=400,         # at most 400 characters ... if possible
    chunk_overlap=0,        # no overlap
)
chunks = by_paragraph.split_text(bio)                          # a list of str
print("chunk lengths:", [len(c) for c in chunks])              # two paragraphs are longer than 400: see the warnings

Created a chunk of size 460, which is longer than the specified 400
Created a chunk of size 520, which is longer than the specified 400


chunk lengths: [460, 329, 520, 377]


The warnings tell us that paragraphs 1 and 3 did not fit, and `CharacterTextSplitter` had no other place to cut. With `separator=" "` it can cut between any two words, so every chunk fits. We also add an overlap of 30 characters (figure above):

In [4]:
by_word = CharacterTextSplitter(
    separator=" ",          # cut between words
    chunk_size=300,         # at most 300 characters
    chunk_overlap=30,       # repeat up to 30 characters of whole words at the start of the next chunk
)
chunks = by_word.split_text(bio)
print("chunk lengths:", [len(c) for c in chunks])               # all at most 300

print("\nend of chunk 1  :", repr(chunks[0][-60:]))              # the border between chunks 1 and 2
print("start of chunk 2:", repr(chunks[1][:60]))                 # begins with the last words of chunk 1

chunk lengths: [295, 297, 292, 293, 299, 293, 77]

end of chunk 1  : 'CIT, University of the Punjab. His research focuses on fuzzy'
start of chunk 2: 'His research focuses on fuzzy inference models applied to op'


Every chunk now fits, and the overlap repeats *"His research focuses on fuzzy"*. But cutting between words does not care about sentences: chunk 1 stops in the middle of one.

## <span style='color:#7e22ce'>Example 3: <code>TokenTextSplitter</code>: counting tokens instead of characters</span>

Models count **tokens**, not characters. `TokenTextSplitter` measures `chunk_size` in tokens with **tiktoken**, the tokenizer of OpenAI models. But every model family has **its own tokenizer**, so the same text has a different number of tokens for each one.

In [5]:
import tiktoken                                                  # OpenAI's tokenizer library
from langchain_text_splitters import TokenTextSplitter

openai_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",    # the tokenizer of GPT-4 and OpenAI's embedding models (default: "gpt2")
    chunk_size=100,                 # at most 100 tiktoken tokens per chunk
    chunk_overlap=0,
)
cl100k = tiktoken.get_encoding("cl100k_base")                    # to count tiktoken tokens ourselves

print("chunk   chars   tiktoken tokens   MiniLM tokens")
for i, c in enumerate(openai_splitter.split_text(bio), 1):
    print(f"{i:5} {len(c):7} {len(cl100k.encode(c)):17} {count_tokens(c):15}")   # same text, different counts

chunk   chars   tiktoken tokens   MiniLM tokens
    1     483               100             102
    2     569               100             107
    3     640                97             110


The same chunk is about 100 tokens for OpenAI but more for MiniLM. **Count with the tokenizer of the model you will use.** For our sentence-transformers model, `SentenceTransformersTokenTextSplitter` does exactly that:

In [6]:
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

minilm_splitter = SentenceTransformersTokenTextSplitter(
    model_name=MODEL,            # count with MiniLM's own tokenizer
    tokens_per_chunk=200,        # at most 200 tokens per chunk: safely below the 256 limit
    chunk_overlap=0,
)
chunks = minilm_splitter.split_text(bio)
print("MiniLM tokens per chunk:", [count_tokens(c) for c in chunks])     # 200 + the 2 special tokens, then the rest
print("\nstart of chunk 1:", repr(chunks[0][:80]))                        # note: all lower case!

MiniLM tokens per chunk: [202, 115]

start of chunk 1: 'dr. muhammad arif butt is an accomplished assistant professor at the department '


Every chunk fits the model now, but look at the text: **all lower case**. This splitter turns the text into token numbers and back, and MiniLM's tokenizer forgets capital letters. The chunks are fine for the model, but not nice to show to a user. Example 5 shows a better way: count MiniLM tokens, but cut the original text.

# <span style='background :lightgreen' >4. Splitting by Text Structure: the Recommended Default</span>

## <span style='color:#7e22ce'>Example 4: <code>RecursiveCharacterTextSplitter</code></span>

The LangChain documentation says: *"For most use cases, start with the RecursiveCharacterTextSplitter."* It has a **list** of separators, from big to small: paragraphs `"\n\n"`, lines `"\n"`, words `" "`, and single characters `""`. It cuts at the biggest separator first, and uses a smaller one **only for the pieces that are still too long**.

<div style="text-align:center;">
    <img src="../images/langchain-new-23.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(
    chunk_size=400,                       # at most 400 characters (default: 4000)
    chunk_overlap=0,                      # no overlap, to see the cuts clearly (default: 200)
    # separators=["\n\n", "\n", " ", ""]  # the default list: paragraphs, lines, words, characters
)
chunks = recursive.split_text(bio)

for i, c in enumerate(chunks, 1):
    print(f"chunk {i}: {len(c):3} characters | {c[:45]!r} ... {c[-25:]!r}")

chunk 1: 399 characters | 'Dr. Muhammad Arif Butt is an accomplished Ass' ... 'ed services, particularly'
chunk 2:  60 characters | 'in decision-making under uncertain and imprec' ... 'and imprecise conditions.'
chunk 3: 329 characters | 'With over 33 years of experience in teaching ' ... ' artificial intelligence.'
chunk 4: 385 characters | 'Beyond academia, he is a technology entrepren' ... 'understanding of software'
chunk 5: 134 characters | 'architectures, memory corruption techniques, ' ... 'e cybersecurity defences.'
chunk 6: 377 characters | 'A dedicated and results-driven professional, ' ... 't evolving cyber threats.'


Compare with Example 2: paragraphs 2 and 4 stay **whole**, and only the two long paragraphs are cut, between words. Every chunk fits in 400 characters.

## <span style='color:#7e22ce'>Example 5: <code>split_documents()</code>: chunks that remember their page</span>

In a RAG app we split the **Documents** from a loader, not strings. `split_documents()` copies each Document's metadata into all its chunks, so every chunk still knows its file and page. We split the 15 pages of `attention.pdf` (Lec-34, Example 4) in two ways:
1. `chunk_size=1000` characters, a common setting in tutorials;
2. `chunk_size=200` **MiniLM tokens**, with `from_huggingface_tokenizer()`: the recursive splitter cuts the original text, but measures each piece with the model's tokenizer.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("../data/attention.pdf").load()            # 15 Documents, one per page (Lec-34)

# 1. measured in characters
by_chars = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = by_chars.split_documents(pages)                       # Documents in, Documents out
too_long = [c for c in chunks if count_tokens(c.page_content) > 256]
print(f"1000 characters: {len(chunks)} chunks, {len(too_long)} of them longer than 256 MiniLM tokens")

# 2. measured in MiniLM tokens
by_tokens = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,              # count length with MiniLM's tokenizer
    chunk_size=200,         # at most 200 tokens per chunk
    chunk_overlap=20,       # about 10% overlap
    add_start_index=True,   # record where each chunk starts in its page
)
chunks = by_tokens.split_documents(pages)
print(f"200 MiniLM tokens: {len(chunks)} chunks, the longest has {max(count_tokens(c.page_content) for c in chunks)} tokens")

1000 characters: 49 chunks, 10 of them longer than 256 MiniLM tokens
200 MiniLM tokens: 60 chunks, the longest has 202 tokens


With 1000 characters, some chunks are longer than 256 tokens: their ends would be cut off silently, as in Example 1. Measuring in the model's tokens guarantees that every chunk fits. Now look at the metadata of one chunk:

In [9]:
chunk = chunks[10]                                                       # any chunk
for key in ["source", "page", "page_label", "start_index"]:
    print(f"{key:12}: {chunk.metadata[key]}")                            # copied from the page, plus start_index
print("\ntext:", chunk.page_content[:150], "...")

start = chunk.metadata["start_index"]                                    # check: the chunk really is at that position
page_text = pages[chunk.metadata["page"]].page_content
print("\nfound at start_index in its page?", page_text[start:start + 50] == chunk.page_content[:50])

source      : ../data/attention.pdf
page        : 2
page_label  : 3
start_index : 770

text: itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel =  ...

found at start_index in its page? True


# <span style='background :lightgreen' >5. Splitting by Document Structure</span>

## <span style='color:#7e22ce'>Example 6: <code>MarkdownHeaderTextSplitter</code>: headings become metadata</span>

Many documents already have a structure: Markdown files, wikis and READMEs have headings. `MarkdownHeaderTextSplitter` cuts at the headings you list, and puts the headings above each piece into its **metadata**. A chunk then knows which section it belongs to, even though the heading text is not in the chunk.

In [10]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

course_notes = """# LangChain Course
Lectures 30 to 38 teach the building blocks of LLM apps.

## Document Loaders
Loaders read files, web pages and databases into Document objects.

## Text Splitters
Splitters cut large Documents into chunks.

### Recursive Splitting
Tries paragraphs first, then lines, then words.

## Vector Stores
A vector store saves each chunk with its embedding.
"""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "course"), ("##", "section"), ("###", "subsection")],  # (heading mark, metadata key)
)
for d in md_splitter.split_text(course_notes):        # returns Documents, not strings
    print(d.metadata, "\n   ->", d.page_content)

{'course': 'LangChain Course'} 
   -> Lectures 30 to 38 teach the building blocks of LLM apps.
{'course': 'LangChain Course', 'section': 'Document Loaders'} 
   -> Loaders read files, web pages and databases into Document objects.
{'course': 'LangChain Course', 'section': 'Text Splitters'} 
   -> Splitters cut large Documents into chunks.
{'course': 'LangChain Course', 'section': 'Text Splitters', 'subsection': 'Recursive Splitting'} 
   -> Tries paragraphs first, then lines, then words.
{'course': 'LangChain Course', 'section': 'Vector Stores'} 
   -> A vector store saves each chunk with its embedding.


A search can now use the metadata, for example *"only chunks from the section Text Splitters"*. If a section is long, split its Documents again with `RecursiveCharacterTextSplitter.split_documents()`: the heading metadata is kept.

## <span style='color:#7e22ce'>Example 7: A PDF with a complex table: Docling</span>

Page 9 of `attention.pdf` holds **Table 3**: a header on two rows, groups of rows labelled (A) to (E), and many **empty cells** (an empty cell means "same as the base model"). First, what `PyPDFLoader` gives us for it:

In [11]:
page9 = pages[8].page_content                     # page 9 of attention.pdf, from Example 5 (lists start at 0)
start = page9.index("N d model")                  # where the table's header starts in the extracted text
print(page9[start:start + 330])                   # the header and the first rows

N d model dff h d k dv Pdrop ϵls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
256 32 32 5.75 24.5 28
1024 128 128


`pypdf` reads the PDF's text **line by line**. The header is spread over three lines, and every **empty cell has disappeared**, so the values slide to the left. In the row `(B) 16 5.16 25.1 58`, is 16 the value of `h`, `dk` or `dv`? Nobody can tell any more, not even a human.

A splitter only decides where to cut **text it is given**, so it cannot bring back what the loader lost. Here is the chunk that holds the (B) rows, from a recursive splitter:

In [12]:
page9_chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0).split_text(page9)
print(next(c for c in page9_chunks if "(B) 16" in c))   # the chunk with the (B) rows: the values no longer line up with the header

per-word perplexities.
N d model dff h d k dv Pdrop ϵls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36


### Docling: read the layout, not just the text
**Docling** (from IBM, open source) looks at the page the way a person does:
- a **layout model** finds the text blocks, headings, figures and tables on the page;
- a **table model** finds the rows, the columns and the merged cells of each table;
- the result is exported as **Markdown**: headings become `#` lines and tables become Markdown tables, with empty cells kept empty.

It runs on your computer and is free. The first time, it downloads its models (about 500 MB); after that it needs about 10 to 15 seconds per page on a laptop CPU. To keep the example short, we give it only page 9, saved as a one-page PDF.

In [13]:
import logging                                                     # to quiet Docling's log messages
from pypdf import PdfReader, PdfWriter                             # the library behind PyPDFLoader
from langchain_docling import DoclingLoader                        # LangChain's loader for Docling
from langchain_docling.loader import ExportType                    # how DoclingLoader returns the result
from IPython.display import Markdown, display                      # to show a Markdown table as a real table

logging.getLogger("docling").setLevel(logging.ERROR)               # hide info messages and a harmless plugin warning

writer = PdfWriter()                                               # a new, empty PDF
writer.add_page(PdfReader("../data/attention.pdf").pages[8])       # copy page 9 into it
writer.write("../data/attention_page9.pdf")                        # a one-page PDF that this notebook creates

loader = DoclingLoader(file_path="../data/attention_page9.pdf",
                       export_type=ExportType.MARKDOWN)            # the whole page as one Markdown Document
page_md = loader.load()[0].page_content                            # slow the first time: the models are downloaded

lines = page_md.splitlines()
table_lines = [l for l in lines if l.startswith("|")]              # Markdown table rows start with "|"
print("table:", len(table_lines), "lines,", count_tokens("\n".join(table_lines)), "MiniLM tokens")
display(Markdown("\n".join(table_lines)))                          # show the table that Docling found

table: 22 lines, 1045 MiniLM tokens


|      | N                                         | dmodel                                    | dff                                       | h                                         | dk                                        | dv                                        | Pdrop                                     | ϵls                                       | train steps   |   PPL (dev) |   BLEU (dev) |   params ×106 |
|------|-------------------------------------------|-------------------------------------------|-------------------------------------------|-------------------------------------------|-------------------------------------------|-------------------------------------------|-------------------------------------------|-------------------------------------------|---------------|-------------|--------------|---------------|
| base | 6                                         | 512                                       | 2048                                      | 8                                         | 64                                        | 64                                        | 0.1                                       | 0.1                                       | 100K          |        4.92 |         25.8 |            65 |
|      |                                           |                                           |                                           | 1                                         | 512                                       | 512                                       |                                           |                                           |               |        5.29 |         24.9 |               |
|      |                                           |                                           |                                           | 4                                         | 128                                       | 128                                       |                                           |                                           |               |        5.00 |         25.5 |               |
| (A)  |                                           |                                           |                                           | 16                                        | 32                                        | 32                                        |                                           |                                           |               |        4.91 |         25.8 |               |
|      |                                           |                                           |                                           | 32                                        | 16                                        | 16                                        |                                           |                                           |               |        5.01 |         25.4 |               |
|      |                                           |                                           |                                           |                                           | 16                                        |                                           |                                           |                                           |               |        5.16 |         25.1 |            58 |
| (B)  |                                           |                                           |                                           |                                           | 32                                        |                                           |                                           |                                           |               |        5.01 |         25.4 |            60 |
|      | 2                                         |                                           |                                           |                                           |                                           |                                           |                                           |                                           |               |        6.11 |         23.7 |            36 |
|      | 4                                         |                                           |                                           |                                           |                                           |                                           |                                           |                                           |               |        5.19 |         25.3 |            50 |
|      | 8                                         |                                           |                                           |                                           |                                           |                                           |                                           |                                           |               |        4.88 |         25.5 |            80 |
| (C)  |                                           | 256                                       |                                           |                                           | 32                                        | 32                                        |                                           |                                           |               |        5.75 |         24.5 |            28 |
|      |                                           | 1024                                      |                                           |                                           | 128                                       | 128                                       |                                           |                                           |               |        4.66 |         26.0 |           168 |
|      |                                           |                                           | 1024                                      |                                           |                                           |                                           |                                           |                                           |               |        5.12 |         25.4 |            53 |
|      |                                           |                                           | 4096                                      |                                           |                                           |                                           |                                           |                                           |               |        4.75 |         26.2 |            90 |
|      |                                           |                                           |                                           |                                           |                                           |                                           | 0.0                                       |                                           |               |        5.77 |         24.6 |               |
|      |                                           |                                           |                                           |                                           |                                           |                                           | 0.2                                       |                                           |               |        4.95 |         25.5 |               |
| (D)  |                                           |                                           |                                           |                                           |                                           |                                           |                                           | 0.0                                       |               |        4.67 |         25.3 |               |
|      |                                           |                                           |                                           |                                           |                                           |                                           |                                           | 0.2                                       |               |        5.47 |         25.7 |               |
| (E)  | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids |               |        4.92 |         25.7 |               |
| big  | 6                                         | 1024                                      | 4096                                      | 16                                        |                                           |                                           | 0.3                                       |                                           | 300K          |        4.33 |         26.4 |           213 |

Every value is back in its own column: in the (B) rows, 16 and 32 are under `dk`, and the "big" row reads 6, 1024, 4096, 16, 0.3, 300K. Two small flaws remain: a **merged cell** is copied into every column it covers (row (E)), and a group label such as "(A)" appears on only one row of its group.

### Splitting the page
The table is **1045 MiniLM tokens**: far too long for one chunk (Example 1). We split the two parts of the page differently:
- **the text**: at the headings with `MarkdownHeaderTextSplitter` (Example 6), then to 200 tokens with the splitter of Example 5;
- **the table**: into groups of whole rows, and every group gets the table's title and the **header row**, so each number keeps its column name. LangChain has no splitter for this, so we write a small function.

In [14]:
from langchain_core.documents import Document           # each chunk is a Document

def split_markdown_table(table_lines: list[str], title: str, max_tokens: int = 250) -> list[Document]:
    """Split a Markdown table into Documents of whole rows; each one repeats the title and the header row."""
    def cells(line): return [c.strip() for c in line.strip().strip("|").split("|")]   # "| a | b |" -> ["a", "b"]
    def row(values): return "| " + " | ".join(values) + " |"                          # ["a", "b"] -> "| a | b |"

    columns = cells(table_lines[0])                                      # the first line holds the column names
    header = row(columns) + "\n" + row(["---"] * len(columns))           # column names + the Markdown separator line
    def make(rows): return title + "\n\n" + header + "\n" + "\n".join(rows)   # the text of one chunk

    groups, current = [], []
    for line in table_lines[2:]:                                         # the data rows (skip header and separator)
        r = row(cells(line))
        if current and count_tokens(make(current + [r])) > max_tokens:   # this row would not fit any more:
            groups.append(current)                                       # close the current group
            current = []
        current.append(r)
    groups.append(current)                                               # the last group
    return [Document(page_content=make(g), metadata={"type": "table", "table": title}) for g in groups]

title = next(l for l in lines if l.startswith("Table 3:")).split(". ")[0] + "."   # the first sentence of the caption
table_chunks = split_markdown_table(table_lines, title)                           # 250: just under MiniLM's 256

text_only = "\n".join(l for l in lines if not l.startswith("|"))                  # the page without the table
sections = MarkdownHeaderTextSplitter([("#", "title"), ("##", "section")]).split_text(text_only)
text_chunks = by_tokens.split_documents(sections)                                 # Example 5's splitter: 200 tokens

print("text chunks :", len(text_chunks), "| MiniLM tokens:", [count_tokens(c.page_content) for c in text_chunks])
print("table chunks:", len(table_chunks), "| MiniLM tokens:", [count_tokens(c.page_content) for c in table_chunks])
print("\nthe last table chunk:")
display(Markdown(table_chunks[-1].page_content))                                  # whole rows, with all the column names

text chunks : 4 | MiniLM tokens: [175, 116, 185, 59]
table chunks: 5 | MiniLM tokens: [243, 224, 233, 183, 248]

the last table chunk:


Table 3: Variations on the Transformer architecture.

|  | N | dmodel | dff | h | dk | dv | Pdrop | ϵls | train steps | PPL (dev) | BLEU (dev) | params ×106 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| (E) | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids | positional embedding instead of sinusoids |  | 4.92 | 25.7 |  |
| big | 6 | 1024 | 4096 | 16 |  |  | 0.3 |  | 300K | 4.33 | 26.4 | 213 |

Every chunk fits the embedding model, and every table chunk can be understood **on its own**: a question such as *"What BLEU score did the big Transformer get?"* can find the last chunk, and the model sees `BLEU (dev)` right above `26.4`.

<div style="text-align:center;">
    <img src="../images/langchain-new-25.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

> **Other tools:** `PyMuPDF4LLMLoader` (lighter, finds tables with rules rather than a model) and **LlamaParse** (a cloud service by LlamaIndex: it needs an API key, and only a limited number of pages are free) also turn PDFs into Markdown. For scanned pages, Docling can also run OCR.

## <span style='color:#7e22ce'>Example 8: Splitting source code</span>

For code, good places to cut are between classes and functions. `RecursiveCharacterTextSplitter.from_language()` uses a separator list made for one programming language. For Python it is `"\nclass "`, `"\ndef "`, `"\n\tdef "`, then blank lines, lines, words and characters. More than 25 languages are supported (`Language.JS`, `Language.JAVA`, `Language.CPP`, `Language.MARKDOWN`, ...).

In [15]:
from langchain_text_splitters import Language

print(RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON))   # the separators for Python

python_code = """
class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

def greet(name):
    return f"Hello, {name}!"

def main():
    calc = Calculator()
    result = calc.add(5, 3)
    print(f"Result: {result}")
    print(greet("World"))
"""

code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,   # use the Python separators
    chunk_size=150,             # big enough for each function in this small file
    chunk_overlap=0,            # functions are self-contained: no overlap needed
)
for i, c in enumerate(code_splitter.split_text(python_code), 1):
    print(f"\n----- chunk {i} ({len(c)} characters) -----")
    print(c)

['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']

----- chunk 1 (115 characters) -----
class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

----- chunk 2 (45 characters) -----
def greet(name):
    return f"Hello, {name}!"

----- chunk 3 (120 characters) -----
def main():
    calc = Calculator()
    result = calc.add(5, 3)
    print(f"Result: {result}")
    print(greet("World"))


Each chunk is one complete class or function. Try `chunk_size=100`: the class no longer fits and is cut between its methods.

# <span style='background :lightgreen' >6. Splitting by Meaning</span>

## <span style='color:#7e22ce'>Example 9: <code>SemanticChunker</code></span>

All the splitters so far look at characters: blank lines, spaces, headings. A **semantic** splitter looks at **meaning**:
1. cut the text into sentences;
2. embed every sentence;
3. measure the **distance** between neighbouring sentences (1 − cosine similarity);
4. cut where the distance is unusually large: above a **threshold**, for example the 75th percentile of all distances.

It is in the `langchain-experimental` package. To test it, we use 9 sentences on three topics written as **one paragraph**: no blank line shows where a topic changes, only the meaning does.

> **Paper:** [Semantic Text Splitting Method Development for RAG](https://github.com/arifpucit/Generative-and-Agentic-AI/blob/main/Research%20Articles/37-Semantic_text_splitting_method_development_for_rag.pdf)

In [16]:
from langchain_experimental.text_splitter import SemanticChunker   # splits by meaning, with any embedding model

mixed_text = (
    "Python is a high-level programming language created by Guido van Rossum. "
    "It was first released in 1991 and is known for its simple, readable syntax. "
    "Today Python is widely used in web development, data science and automation. "
    "Cricket is a bat-and-ball sport played between two teams of eleven players. "
    "A Test match can last up to five days, while a T20 game finishes in about three hours. "
    "Pakistan won the Cricket World Cup in 1992 under the captaincy of Imran Khan. "
    "Machine learning is a branch of artificial intelligence that learns patterns from data. "
    "In supervised learning, a model is trained on examples that already have the correct labels. "
    "Common applications include spam filters, image recognition and recommendation systems."
)

def show(chunks):
    """Print each chunk's first words."""
    for i, c in enumerate(chunks, 1):
        print(f"  chunk {i}: {c[:70]}...")

# 1. default settings
default = SemanticChunker(embeddings)                 # buffer_size=1, threshold = 95th percentile
print("default settings:")
show(default.split_text(mixed_text))

default settings:
  chunk 1: Python is a high-level programming language created by Guido van Rossu...
  chunk 2: In supervised learning, a model is trained on examples that already ha...


With the default settings we get 2 chunks, and the cut is in the wrong place: inside the machine-learning sentences, not at "Cricket" or "Machine learning". Two settings matter:
- **`buffer_size`** (default 1): each sentence is embedded **together with its neighbours**. This smooths the distances, which can hide a sharp change. With `0`, each sentence is embedded alone.
- **`breakpoint_threshold_amount`** (default 95 for `"percentile"`): cut only where the distance is in the top 5%. With 8 distances, that is at most one cut. A **lower** amount gives **more** cuts.

In [17]:
tuned = SemanticChunker(
    embeddings,
    buffer_size=0,                          # embed each sentence alone
    breakpoint_threshold_type="percentile", # the threshold is a percentile of all distances
    breakpoint_threshold_amount=75,         # cut where the distance is in the top 25%
)
print("tuned settings:")
show(tuned.split_text(mixed_text))

tuned settings:
  chunk 1: Python is a high-level programming language created by Guido van Rossu...
  chunk 2: Cricket is a bat-and-ball sport played between two teams of eleven pla...
  chunk 3: Machine learning is a branch of artificial intelligence that learns pa...


Now each chunk is one topic. The figure shows the 8 real distances: the two largest are exactly where the topic changes.

<div style="text-align:center;">
    <img src="../images/langchain-new-24.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**Two weaknesses to remember:**
- The best settings depend on the text and the embedding model: **tune them on your own documents.** In Chroma's chunking study (2024), `SemanticChunker` with default settings scored slightly *below* average.
- It finds sentences with a simple rule, *"a `.`, `?` or `!` followed by a space"*, so abbreviations break sentences:

In [18]:
import re                                                   # SemanticChunker's sentence rule, in one line

sentence_rule = r"(?<=[.?!])\s+"                            # a space that follows . ? or !
print(re.split(sentence_rule, "He holds a Ph.D. in Computer Science from PUCIT. His research focuses on fuzzy models."))

['He holds a Ph.D.', 'in Computer Science from PUCIT.', 'His research focuses on fuzzy models.']


## <span style='color:#7e22ce'>Example 10: Asking an LLM where to cut</span>

An LLM *understands* the text, so it can find topic changes without any threshold. We ask the free Groq model to return the chunks as a list, using `with_structured_output()` (Lec-30, Example 9).

An LLM may also **change** the text while copying it: fix a word, shorten a sentence. Chunks must be the original text, so we **check** that each chunk appears word for word in the input.

In [19]:
import os                                                    # read GROQ_API_KEY
from dotenv import load_dotenv                               # load the API keys from ../keys/.env
from pydantic import BaseModel, Field                        # describe the output we want
from langchain_openai import ChatOpenAI                      # works with Groq's OpenAI-compatible API (Lec-30)

load_dotenv("../keys/.env", override=True)
print("GROQ_API_KEY", "set" if os.getenv("GROQ_API_KEY") else "MISSING")   # never print any part of a key
GROQ_MODEL = ChatOpenAI(model="openai/gpt-oss-120b", base_url="https://api.groq.com/openai/v1",
                        api_key=os.getenv("GROQ_API_KEY"), max_retries=5)  # free

class Chunk(BaseModel):
    topic: str = Field(description="a short title for this chunk, 2 to 5 words")
    text: str = Field(description="the sentences of this chunk, copied word for word from the input")

class Chunks(BaseModel):
    chunks: list[Chunk]                                      # the model returns a list of Chunk objects

chunker = GROQ_MODEL.with_structured_output(Chunks)          # a model that returns a Chunks object

result = chunker.invoke(
    "Split the text below into chunks, one chunk per topic. "
    "Copy the sentences exactly; do not change, add or remove any word.\n\nTEXT:\n" + mixed_text
)
for c in result.chunks:
    print(f"{c.topic:20} | copied word for word: {c.text in mixed_text} | {c.text[:50]}...")

GROQ_API_KEY set
Python programming language | copied word for word: True | Python is a high-level programming language create...
Cricket sport        | copied word for word: True | Cricket is a bat-and-ball sport played between two...
Machine learning     | copied word for word: True | Machine learning is a branch of artificial intelli...


The LLM found the three topics and gave each one a title, which could go into the metadata. In Chroma's study an LLM chunker had the **highest recall** of all methods. But it is the **slowest and most expensive**: one call per piece of text, and long documents must be sent in parts. Use it when the quality of the chunks matters more than the cost.

# <span style='background :lightgreen' >7. Which Splitter Should I Use?</span>

| Your content | Splitter | Example |
|---|---|---|
| Most text: articles, PDFs, web pages | `RecursiveCharacterTextSplitter`, sized in your embedding model's tokens | 4, 5 |
| Markdown, wikis, READMEs | `MarkdownHeaderTextSplitter`, then the recursive splitter for long sections | 6 |
| HTML pages | `HTMLHeaderTextSplitter` (same idea as Markdown) | - |
| PDFs with complex tables or scanned pages | Docling to Markdown; split the text by headings and the tables by rows | 7 |
| Source code | `RecursiveCharacterTextSplitter.from_language(...)` | 8 |
| Topics change without any visible structure | `SemanticChunker` (tune it), or an LLM if quality matters most | 9, 10 |
| Short texts: tweets, FAQ answers, table rows | **do not split**: each one is already a good chunk | Lec-34 |

**Size and overlap:** Chroma's study (*Evaluating Chunking Strategies for Retrieval*, 2024, https://www.trychroma.com/research/evaluating-chunking) found that:
- `RecursiveCharacterTextSplitter` with **about 200 to 400 tokens** and **little or no overlap** works well;
- OpenAI's old default of 800 tokens with 400 overlap scored poorly.

Start there, never exceed your embedding model's limit, and then test with real questions on your own documents (Lec-37).

# <span style='background :lightgreen' >8. Summary</span>

| Ex | Splitter | Cuts at | Package |
|---|---|---|---|
| 2 | `CharacterTextSplitter` | one separator only | `langchain-text-splitters` |
| 3 | `TokenTextSplitter`, `SentenceTransformersTokenTextSplitter` | a number of tokens | `langchain-text-splitters` (+ `tiktoken`, `sentence-transformers`) |
| 4, 5 | `RecursiveCharacterTextSplitter` | paragraphs, then lines, then words | `langchain-text-splitters` |
| 6 | `MarkdownHeaderTextSplitter` | headings, kept as metadata | `langchain-text-splitters` |
| 7 | `DoclingLoader` + `MarkdownHeaderTextSplitter` + our table helper | headings, and groups of table rows | `langchain-docling`, `docling` |
| 8 | `RecursiveCharacterTextSplitter.from_language()` | classes and functions | `langchain-text-splitters` |
| 9 | `SemanticChunker` | large jumps in meaning | `langchain-experimental` |
| 10 | an LLM with structured output | topics the LLM finds | any chat model |

### Key takeaways
- Embedding models read a limited number of tokens (**256** for MiniLM) and ignore the rest **silently**: split before you embed.
- `chunk_size` is the maximum length; `chunk_overlap` repeats the end of one chunk at the start of the next.
- Count length with the **tokenizer of your embedding model**: `from_huggingface_tokenizer()` does it while keeping the original text.
- Start with **`RecursiveCharacterTextSplitter`**: it keeps paragraphs whole when they fit.
- Use **`split_documents()`** in a RAG app: every chunk keeps the metadata (file, page) of its Document.
- Structure-aware splitters use headings or code blocks; semantic and LLM splitters use meaning, but need tuning or cost more.
- A splitter cannot repair a badly loaded table: load complex PDFs with a layout-aware tool such as **Docling**, and split tables by rows with the column names in every chunk.
- **Next lecture (Lec-36):** **Vector stores**: embed the chunks and save them for search.

### Check your understanding
1. In Example 1, why did the similarity stay at 1.0000 when we added a sentence at the end, but not at the start?
2. What do `chunk_size` and `chunk_overlap` mean? What problem does overlap solve, and what does it cost?
3. Why did `CharacterTextSplitter(separator="\n\n", chunk_size=400)` produce chunks longer than 400?
4. Why does the same chunk have a different number of tokens for tiktoken and for MiniLM?
5. How does `RecursiveCharacterTextSplitter` decide where to cut? Why is chunk 2 of Example 4 only 60 characters?
6. What is the difference between `split_text()` and `split_documents()`? Which one would you use in a RAG app, and why?
7. What does `add_start_index=True` add, and how could you use it?
8. What does `MarkdownHeaderTextSplitter` put into the metadata?
9. How does `SemanticChunker` decide where to cut? What do `buffer_size` and `breakpoint_threshold_amount` change?
10. In Example 7, why can no splitter repair the text that `PyPDFLoader` gives for Table 3? Why do we repeat the column names in every table chunk?
11. Why did we check `c.text in mixed_text` in Example 10?